# 05.2 Model Comparison: Baseline vs GRU vs GBDT vs CatBoost

Comparación reproducible entre los modelos actuales del repo usando los artefactos ya generados:
- `market_value_baseline`
- `price_sequence_gru`
- `hist_gradient_boosting`
- `catboost_residual`

El primario se elige por:
1. `top_k_avg_realized_pnl`
2. `brier`
3. `log_loss`
con la restricción de que debe superar al benchmark de mercado en al menos una métrica probabilística.


## 1. Setup

Cargamos raw data, checkpoints y pipeline del baseline. Si alguno de estos artefactos falta, la comparación todavía no está lista.

In [1]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.config import load_config

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

cfg = load_config(str(ROOT / 'config' / 'config.yaml'))
MODELS = ROOT / 'data' / 'models'
REGISTRY = MODELS / 'registry'
V1 = MODELS / 'v1'
V1_REGISTRY = V1 / 'registry'
comparison = json.loads((REGISTRY / 'model_comparison.json').read_text())
primary = json.loads((REGISTRY / 'primary_model.json').read_text())
live_summary = json.loads((REGISTRY / 'live_scoring_summary.json').read_text())
live_df = pd.read_csv(REGISTRY / 'live_scores.csv')
v1_comparison = json.loads((V1_REGISTRY / 'model_comparison.json').read_text()) if (V1_REGISTRY / 'model_comparison.json').exists() else None


## 2. Definición del universo común de evaluación

Primero leemos las métricas held-out guardadas por cada entrenamiento.
Después construimos un **universo común de diagnóstico** usando mercados que el TS sí puede secuenciar; eso no sustituye el test puro, pero sí permite comparar scores lado a lado.


In [2]:
rows = []
for model_name, payload in comparison['models'].items():
    test = payload['test']
    rows.append({
        'model_name': model_name,
        'beats_market': payload.get('beats_market'),
        'brier': test['calibrated_metrics']['brier'],
        'market_brier': test['market_baseline_metrics']['brier'],
        'log_loss': test['calibrated_metrics']['log_loss'],
        'market_log_loss': test['market_baseline_metrics']['log_loss'],
        'top_k_avg_realized_pnl': test['ev_metrics']['top_k_avg_realized_pnl'],
        'top_k_hit_rate': test['ev_metrics']['top_k_hit_rate'],
        'live_buy_signals': live_summary.get(model_name, {}).get('signals', {}).get('BUY', 0) + live_summary.get(model_name, {}).get('signals', {}).get('STRONG BUY', 0),
    })
summary_df = pd.DataFrame(rows).sort_values('top_k_avg_realized_pnl', ascending=False)
display(summary_df.round(4))


,model_name,beats_market,brier,market_brier,log_loss,market_log_loss,top_k_avg_realized_pnl,top_k_hit_rate,live_buy_signals
3,catboost_residual,True,0.0042,0.0062,0.0143,0.0221,0.1129,0.99,0
2,hist_gradient_boosting,True,0.0046,0.0062,0.0177,0.0221,0.1102,0.77,13
1,price_sequence_gru,False,0.0269,0.0062,0.1287,0.0221,-0.0268,0.00,52
0,market_value_baseline,False,0.0111,0.0062,0.0462,0.0221,-0.0486,0.05,40


## 3. Carga de ambos modelos

Aquí no entrenamos nada: solo cargamos los checkpoints ya producidos por `04` y `04_1`.

In [3]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.bar(summary_df['model_name'], summary_df['top_k_avg_realized_pnl'], color=PALETTE[:len(summary_df)], alpha=0.85)
ax.axhline(0, color='black', lw=1)
ax.set_title('Top-K avg realized PnL')
ax.tick_params(axis='x', rotation=20)

ax = axes[0, 1]
width = 0.35
x = np.arange(len(summary_df))
ax.bar(x - width/2, summary_df['brier'], width=width, color=PALETTE[0], label='model')
ax.bar(x + width/2, summary_df['market_brier'], width=width, color=PALETTE[1], label='market')
ax.set_xticks(x)
ax.set_xticklabels(summary_df['model_name'], rotation=20)
ax.set_title('Brier vs mercado')
ax.legend()

ax = axes[1, 0]
ax.bar(summary_df['model_name'], 100 * summary_df['top_k_hit_rate'], color=PALETTE[:len(summary_df)], alpha=0.85)
ax.set_title('Top-K hit rate')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=20)

ax = axes[1, 1]
ax.bar(summary_df['model_name'], summary_df['live_buy_signals'], color=PALETTE[:len(summary_df)], alpha=0.85)
ax.set_title('Señales live accionables')
ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


/var/folders/dv/l82lzhjj64v3xgj4xs_hdqn80000gn/T/ipykernel_26775/529107784.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Scoring del subconjunto temporal compartido

`PriceSequenceGRU` y `MarketValueNet` se evalúan sobre los mismos mercados resueltos y el mismo cutoff temporal. La intersección exacta es la base de la comparación offline.

In [4]:
bucket_rows = []
for model_name, payload in comparison['models'].items():
    for bucket_name, bucket_payload in payload['test'].get('by_horizon', {}).items():
        bucket_rows.append({
            'model_name': model_name,
            'bucket': bucket_name,
            'count': bucket_payload.get('count', 0),
            'brier': bucket_payload.get('probability_metrics', {}).get('brier'),
            'top_k_avg_realized_pnl': bucket_payload.get('ev_metrics', {}).get('top_k_avg_realized_pnl'),
            'top_k_hit_rate': bucket_payload.get('ev_metrics', {}).get('top_k_hit_rate'),
        })
bucket_df = pd.DataFrame(bucket_rows)
display(bucket_df.round(4))


,model_name,bucket,count,brier,top_k_avg_realized_pnl,top_k_hit_rate
0,market_value_baseline,short_1_3d,2380,0.0092,-0.1342,0.03
1,market_value_baseline,medium_4_14d,2380,0.0097,-0.1176,0.02
2,market_value_baseline,long_15plus,1167,0.0177,-0.0311,0.10
3,price_sequence_gru,short_1_3d,2380,0.0243,-0.0078,0.00
4,price_sequence_gru,medium_4_14d,2380,0.0255,-0.0098,0.00
5,price_sequence_gru,long_15plus,1167,0.0352,-0.0102,0.00
6,hist_gradient_boosting,short_1_3d,2380,0.0036,0.0112,0.39
7,hist_gradient_boosting,medium_4_14d,2380,0.0034,0.0139,0.40
8,hist_gradient_boosting,long_15plus,1167,0.0089,0.0829,0.62
9,catboost_residual,short_1_3d,2380,0.0032,0.0255,1.00


## 5. Métricas comparables en validación

Calculamos `AUC-ROC`, `PR-AUC` y `accuracy` para ambos modelos sobre la misma intersección de mercados válidos.

In [5]:
if not bucket_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    pivot = bucket_df.pivot(index='bucket', columns='model_name', values='top_k_avg_realized_pnl')
    pivot.plot.bar(ax=axes[0])
    axes[0].axhline(0, color='black', lw=1)
    axes[0].set_title('Top-K avg realized PnL por bucket')
    axes[0].set_ylabel('avg realized pnl')

    pivot_hit = bucket_df.pivot(index='bucket', columns='model_name', values='top_k_hit_rate')
    (100 * pivot_hit).plot.bar(ax=axes[1])
    axes[1].set_title('Top-K hit rate por bucket')
    axes[1].set_ylabel('hit rate (%)')
    plt.tight_layout()
    plt.show()


/var/folders/dv/l82lzhjj64v3xgj4xs_hdqn80000gn/T/ipykernel_26775/2356585295.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Lectura visual de desempeño offline

Este bloque ayuda a responder dos preguntas:
- cuál modelo se ve mejor en las métricas básicas
- cuánto se parecen realmente sus scores mercado por mercado

In [6]:
active_rows = []
for model_name in comparison['models'].keys():
    sub = live_df[live_df['model_name'] == model_name].copy()
    top = sub[sub['signal'] != 'HOLD'].copy().head(20)
    active_rows.append({
        'model_name': model_name,
        'signals': len(sub[sub['signal'] != 'HOLD']),
        'avg_ev_live': float(sub['ev_per_share'].mean()),
        'top20_avg_ev': float(top['ev_per_share'].mean()) if len(top) else np.nan,
    })
active_compare = pd.DataFrame(active_rows)
display(active_compare.round(4))


,model_name,signals,avg_ev_live,top20_avg_ev
0,market_value_baseline,40,0.2813,0.2381
1,price_sequence_gru,52,0.1038,0.2312
2,hist_gradient_boosting,13,-0.0171,0.0866
3,catboost_residual,0,-0.0307,NaN


## 7. Comparación en mercados activos

Después de la comparación offline, llevamos ambos modelos al mismo snapshot local de activos y aplicamos las mismas reglas de señales. Eso permite medir overlap real de ideas operativas.

In [7]:
actionable_frames = []
for model_name in comparison['models'].keys():
    sub = live_df[(live_df['model_name'] == model_name) & (live_df['signal'] != 'HOLD')][['id', 'question', 'ev_per_share', 'signal']].copy()
    sub = sub.rename(columns={
        'question': f'question_{model_name}',
        'ev_per_share': f'ev_{model_name}',
        'signal': f'signal_{model_name}',
    })
    actionable_frames.append(sub)

if actionable_frames:
    disagreements = actionable_frames[0]
    for frame in actionable_frames[1:]:
        disagreements = disagreements.merge(frame, on='id', how='outer')
    question_cols = [c for c in disagreements.columns if c.startswith('question_')]
    disagreements['question'] = disagreements[question_cols].bfill(axis=1).iloc[:, 0]
    disagreements = disagreements.drop(columns=question_cols)
    ev_cols = [c for c in disagreements.columns if c.startswith('ev_')]
    disagreements['spread_ev'] = disagreements[ev_cols].max(axis=1, skipna=True) - disagreements[ev_cols].min(axis=1, skipna=True)
    disagreements = disagreements.sort_values(['spread_ev', 'question'], ascending=[False, True])
    display(disagreements.head(30))
else:
    print('No hay señales accionables en ningún modelo.')

,id,ev_market_value_baseline,signal_market_value_baseline,ev_price_sequence_gru,signal_price_sequence_gru,ev_hist_gradient_boosting,signal_hist_gradient_boosting,ev_catboost_residual,signal_catboost_residual,question,spread_ev
51,1823795,0.043763,BUY,0.212612,STRONG BUY,NaN,NaN,NaN,NaN,"Will Ethereum reach $2,800 in April?",0.168849
29,1712294,0.229432,STRONG BUY,0.196917,STRONG BUY,0.063390,BUY,NaN,NaN,Will WTI Crude Oil (WTI) hit (HIGH) $150 in Ap...,0.166042
10,1569627,0.149636,STRONG BUY,0.216917,STRONG BUY,0.055604,BUY,NaN,NaN,US x Iran ceasefire by April 15?,0.161312
17,1641031,0.137326,STRONG BUY,0.206917,STRONG BUY,0.090222,STRONG BUY,NaN,NaN,Will another country conduct military action a...,0.116694
0,951183,0.129432,STRONG BUY,0.096917,STRONG BUY,0.031942,BUY,NaN,NaN,Will the Bharatiya Janata Party (BJP) win the ...,0.097490
41,1807966,0.294812,STRONG BUY,0.202112,STRONG BUY,NaN,NaN,NaN,NaN,Will WTI Crude Oil (WTI) hit (HIGH) $170 in Ap...,0.092701
28,1708576,NaN,NaN,0.114796,STRONG BUY,0.045507,BUY,NaN,NaN,Will the Kharg Island oil terminal be hit by A...,0.069289
5,1540766,0.149636,STRONG BUY,0.216917,STRONG BUY,NaN,NaN,NaN,NaN,Strait of Hormuz traffic returns to normal by ...,0.067281
22,1693042,0.059636,BUY,0.126917,STRONG BUY,NaN,NaN,NaN,NaN,Will UAE strike Iran by April 30?,0.067281
34,1763610,0.167636,STRONG BUY,0.234917,STRONG BUY,NaN,NaN,NaN,NaN,US announces military support of Kurds in Iran...,0.067281


## 8. Overlap y desacuerdos más fuertes

La parte más útil de esta sección no es solo el overlap, sino los mercados donde ambos modelos discrepan mucho. Ahí suele estar la señal nueva del modelo secuencial.

In [8]:
print('Modelo primario actual:')
display(pd.Series(primary))

best_model = summary_df.sort_values(['top_k_avg_realized_pnl', 'brier', 'log_loss'], ascending=[False, True, True]).iloc[0]
print('\nLectura rápida:')
print('- Si un modelo no mejora a mercado en Brier o log_loss, no debería ser primario.')
print(f"- Hoy el líder offline es {best_model['model_name']} con Top-K PnL={best_model['top_k_avg_realized_pnl']:.4f} y Brier={best_model['brier']:.4f}.")
if (active_compare['signals'] == 0).any():
    zero_signal_models = ', '.join(active_compare.loc[active_compare['signals'] == 0, 'model_name'])
    print(f"- En live hay modelos extremadamente conservadores sin señales accionables: {zero_signal_models}.")
print('- La comparación ya separa mejor el modelo que gana offline del modelo que realmente produce ideas accionables en activos.')

Modelo primario actual:


name                                                catboost_residual
save_dir                                data/models/catboost_residual
selection_metric    eligible_if(top_k_pnl>0 and beats_market_prob)...
dtype: str


Lectura rápida:
- Si un modelo no mejora a mercado en Brier o log_loss, no debería ser primario.
- Hoy el líder offline es catboost_residual con Top-K PnL=0.1129 y Brier=0.0042.
- En live hay modelos extremadamente conservadores sin señales accionables: catboost_residual.
- La comparación ya separa mejor el modelo que gana offline del modelo que realmente produce ideas accionables en activos.


## 9. Cómo interpretar el resultado final

Al cerrar este notebook, una lectura útil es:
- cuál modelo domina **offline** en Brier, log-loss y Top-K realizado
- cuál modelo todavía genera **señales live** con los thresholds actuales
- dónde discrepan más los modelos en mercados activos

Con el cuarto modelo ya integrado, ahora sí tienes base real para decidir entre:
- un primario único
- un ensemble posterior
- o reglas distintas para offline ranking vs live deployment
